# 07 · Metals

A metal is the same `embed`, given an arrangement. `rx.metal(source, geometry)` enumerates the distinct
coordination isomers; pick one and embed it. Examples use ionic dative SMILES (donor arrows point to the
metal, anionic donors carry formal charges); `rx.metal` also accepts neutral/covalent forms such as
`N->[Pt](Cl)Cl`, preferring the written form `N->[Pt+2](<-[Cl-])<-[Cl-]`.

The force field uses a bond-less metal surrogate for typing; the public graph keeps the real metal and
dative bonds.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import RDLogger
from rdkit.Chem import rdMolTransforms as T

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

## Coordination isomers: the permutations

Square-planar MA₂B₂ gives cis and trans; octahedral MA₃B₃ gives mer and fac. `.summary()` prints each
per-vertex arrangement and Λ/Δ label. Pick one with `.select(index=...)`, `.select(arrangement=...)`, or
`.select(hand=...)`. Arrangement is unambiguous when several isomers share a coarse label.

In [ ]:
print("square-planar Pd (cis/trans):")
rx.metal("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", "square_planar").summary()
print("\noctahedral Co(NH3)3Cl3 (mer/fac):")
rx.metal("N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]", "octahedral").summary()

`screen=False` skips the coordination/ligand compatibility model, not symmetry reduction.
This bipyridyl chelate gains a distinct trans assignment. It is an exploration candidate, not a
promise of successful embedding; the usual geometry and stereo checks still apply.


In [ ]:
bipy = "[Cl-]->[Ni+2]1(<-[Br-])<-[n]2ccccc2-c2cccc[n]->12"
screened = rx.metal(bipy, "SPL")
unscreened = rx.metal(bipy, "SPL", screen=False)
print(f"screened: {len(screened)} isomer(s) | unscreened: {len(unscreened)} isomer(s)")
print("screened subset of unscreened:",
      {rx.cxsmiles(iso) for iso in screened} < {rx.cxsmiles(iso) for iso in unscreened})
unscreened.summary(details=True)

In [ ]:
# The same short-chelate check applies beside a sigma donor or an independent pi ligand.
diamines = [
    "[Cl-]->[Pt+2]1(<-[F-])<-[NH2]CCC[NH2]->1",
    "[Cl-]->[Pt+2]12(<-[NH2]CCC[NH2]->1)<-[CH2]=[CH2]->2",
]
for diamine in diamines:
    cis = rx.metal(diamine, "SPL")
    unscreened = rx.metal(diamine, "SPL", screen=False)
    print(f"{diamine}: screened {len(cis)} | unscreened {len(unscreened)}")
    for iso in cis:
        ensemble = rx.embed(iso, n=1, seed=42, threads=1)
        print(rx.geom_check.check(ensemble.mol, donors=iso.donors).summary())
        print("CX match:", rx.cxsmiles(ensemble.mol) == rx.cxsmiles(iso))

The fac Co isomer, its coordination sphere drawn as dotted dative bonds.


In [ ]:
co = rx.metal("N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]", "octahedral").select(
    index=1
)  # the fac isomer (index 1 in .summary())
ens = rx.embed(co).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    print(report.summary())
xyzrender.render(ens.mol, no_hy=True, idx="s")

The trans Pd isomer: the two Cl sit about 180° apart across the metal.


In [ ]:
trans = rx.metal("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", "square_planar").select(index=1)  # the trans isomer (index 1)
ens = rx.embed(trans).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    print(report.summary())
cl = [d for d in trans.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "Cl"]
print(f"{len(ens)} conformers | Cl-Pd-Cl = {ens.measure((cl[0], trans.metal, cl[1]))['mean']:.0f}° (trans)")
xyzrender.render(
    ens.mol, no_hy=True, labels=[f"{cl[0] + 1} {trans.metal + 1} {cl[1] + 1} a"]
)  # label the trans Cl-Pd-Cl angle

## Across coordination geometries

The same API covers linear, tetrahedral, square-planar, octahedral, trigonal-bipyramidal and
square-pyramidal geometries.

In [ ]:
for g, smi in {
    "linear": "[Cl-]->[Au+]<-[Cl-]",
    "tetrahedral": "[Cl-]->[Zn+2](<-[Cl-])(<-[Cl-])<-[Cl-]",
    "square_planar": "CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC",
    "octahedral": "N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]",
}.items():
    isos = rx.metal(smi, g)
    print(f"  {g:14s}: {len(isos)} isomer(s) {[i.label for i in isos]}")

## Optional UFF cleanup trajectory

Capture one cleanup only when requested: the seed plus retained restrained-UFF snapshots on the public graph.

In [ ]:
iso = rx.metal("C[P]1(C)CC[P](C)(C)->[Ni+2]<-12<-[O-]C(=O)C[N-]->2C", "SPL")[0]
walk = rx.embed(iso, n=1, seed=19, trajectory=True)
print(rx.geom_check.check(walk.mol).summary())
cleanup = walk.trajectory  # RDKit Mol; seed followed by cleanup frames
walk.dump_trajectory("cleanup.xyz")

## Metal-centre chirality

rxembed names Λ and Δ independently of atom order. Each requested hand remains separate throughout embedding.


In [ ]:
isos = rx.metal("[Cl-]->[Co+2]12(<-[Cl-])(<-NCCN->1)<-NCCN->2", "octahedral")  # [Co(en)2Cl2]; bis-chelate + 2 Cl
isos.summary()  # trans | achiral ;  cis | Δ ;  cis | Λ
delta = isos.select(hand="Δ")  # pick the Δ enantiomer by its handedness (name-agnostic, unambiguous)
ens = rx.embed(delta, n=8).mc(preset="ensemble").minimize().prune()
reps = ens.representatives()
print(f"cis-Δ [Co(en)2Cl2]: {ens.n} conformers -> {reps.n} distinct mode(s)")
for report in reps.check(donors=list(delta.donors)).values():
    print(report.summary())  # includes folded-donor checks
xyzrender.render(reps.mol, no_hy=True, idx="s")

## Chelating ligands: bidentate

A chelate's donors sit on one ligand fragment; rxembed widens their L–M–L window for a rigid backbone bite,
while still separating cis from trans. Ethylenediamine on Pd (5-membered ring, N–Pd–N bite ~82°) + two
chloride:

In [ ]:
en = rx.metal("[Cl-]->[Pd+2]1(<-[Br-])<-NCCN->1", "square_planar")  # the NCCN ring pins both N to Pd
iso = en.select()
ens = rx.embed(iso, n=10).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    print(report.summary())
nn = [d for d in iso.donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "N"]
print(f"{len(ens)} conformers | en bite N-Pd-N = {ens.measure((nn[0], iso.metal, nn[1]))['mean']:.0f}°")
xyzrender.render(ens.mol, no_hy=True)

### Two short bites around a tetrahedral centre

The surrounding angle preferences accommodate two disjoint bites together, including pairs joined
by a longer macrocyclic backbone. This does not certify whole-ligand feasibility: ligand bonds and
the selected metal hand still need checking after relaxation.

In [ ]:
smiles = [
    "C=[N]1Cc2cccc[n]2->[Cu+]<-12<-[N](Cc1cccc[n]->21)=C",
    "C1CC[S]2->[Cd+2]34<-[S](C1)CC[S]->3CCCC[S]->4CC2",
]
for smi in smiles:
    isos = rx.metal(smi, "tetrahedral")
    print(f"{smi}: {len(isos)} isomer(s)")
    for iso in isos:
        ens = rx.embed(iso, n=1, seed=42, threads=1)
        print(rx.geom_check.check(ens.mol).summary())
        print("relaxed:", not ens.unrelaxed, "| CX match:", rx.cxsmiles(ens.mol) == rx.cxsmiles(iso))
        print(iso)
        display(xyzrender.render(ens.mol, no_hy=True))

### A tridentate ligand with one axial arm

A TBP equatorial bite can deform while a linked axial donor stays perpendicular to its plane; surrounding
angle targets share that deformation rather than forcing the ideal polyhedron. This selects a seating by
donor slots, not enumeration index, and doesn't establish every alternative is feasible.

In [ ]:
tridentate = rx.metal("[NH2]1CC[NH]2CC[NH2]->[Cd+2](<-[Cl-])(<-[Cl-])<-1<-2", "trigonal_bipyramidal")
# N0 is terminal and axial; central N3 and terminal N6 are equatorial.
iso = next(candidate for candidate in tridentate
           if 0 in candidate.vertices[:2] and {3, 6} <= set(candidate.vertices[2:]))
ens = rx.embed(iso, n=1, seed=42)
print(rx.geom_check.check(ens.mol).summary())
print("relaxed:", not ens.unrelaxed)
print(rx.cxsmiles(ens.mol))

### Two linked bites in a square-pyramidal shell

The same ligand can place its central donor apically, with its terminal donors on opposite sides.
The bite and spectator targets are constructed together. This selected-state example does not certify every enumerated alternative.

In [ ]:
fans = rx.metal("[NH2]1CC[NH]2CC[NH2]->[Cd+2](<-[Cl-])(<-[Cl-])<-1<-2", "square_pyramidal")
# Central N3 is apical; terminal N0/N6 occupy opposite basal slots.
iso = next(candidate for candidate in fans
           if candidate.vertices[0] == 3 and set(candidate.vertices[1::2]) == {0, 6})
ens = rx.embed(iso, n=1, seed=42)
print(rx.geom_check.check(ens.mol).summary())
print("relaxed:", not ens.unrelaxed, "| CX match:", rx.cxsmiles(ens.mol) == rx.cxsmiles(iso))
print(rx.cxsmiles(ens.mol))

## Side-on η² coordination

A directly π-bonded donor pair binds side-on as one haptic site. rxembed holds the donor bond while applying
both metal distances. The Ni example below keeps its coordinated C=C intact.

In [ ]:
iso = rx.metal(
    "COC(=O)[C]12->[Ni+2]3(<-[O-]C(=O)C(c4ccccc4)[N-]->3c3ccccc3)<-[C]=1(C(=O)OC)C2(C)C(C)(C)C",
    "square_planar",
).select(index=0)
pi = next(  # the directly π-bonded donor pair: the side-on C=C
    (a, b)
    for a in iso.donors
    for b in iso.donors
    if a < b and (bd := iso.mol.GetBondBetweenAtoms(a, b)) and bd.GetBondTypeAsDouble() >= 2
)
ens = rx.embed(iso, n=8).mc(preset="ensemble").minimize().prune()
ens.filter("geometry")
c = ens.mol.GetConformer(ens.ids[0])
print(
    f"side-on C=C {pi}: {ens.n} confs | Ni-C {T.GetBondLength(c, iso.metal, pi[0]):.2f}/"
    f"{T.GetBondLength(c, iso.metal, pi[1]):.2f} Å (both bind) | C=C {T.GetBondLength(c, *pi):.2f} Å (held intact)"
)
xyzrender.render(ens.mol, no_hy=True, labels=[f"{iso.metal + 1} {pi[0] + 1} d", f"{iso.metal + 1} {pi[1] + 1} d"])

## Two metal centres in one enumeration

A geometry defines its ligand and haptic stereo, so `rx.metal` retains those states and enumerates both
metals' arrangements by default; name a centre to vary only that one. The bound amine N5 is a stereocentre
only while bound: `stereo={"locked": "racemic"}` gives both hands with every arrangement. CXSMILES records
one selected state, so reloading returns one combination.

In [ ]:
mnh_isomers = rx.metal("structures/mnh.xyz")
print(f"{len(mnh_isomers)} Mn arrangements (9 mer + 6 fac expected); measured Fe, N, and C stereo retained")
mnh_isomers.summary()
mer = mnh_isomers.filter(center="Mn", label="mer")
print(f"{len(mer)} meridional arrangements (donor ordering distinguishes them, so hand alone is not unique)")
chosen = mer.select(hand="lambda", index=4)
chosen.summary(details=True)
print("N5 racemic:", len(rx.metal("structures/mnh.xyz", stereo={"N5": "racemic"})), "arrangements")
print("locked racemic:", len(rx.metal("structures/mnh.xyz", stereo={"locked": "racemic"})), "arrangements")

### Store and embed one selected state

The CXSMILES carries both metal arrangements and the selected ligand stereochemistry.

In [ ]:
written = rx.cxsmiles(chosen)
loaded = rx.metal(written, center="all")
loaded.summary(details=True)
print(f"{len(loaded)} isomer(s) loaded | round-trips to the written CXSMILES:", rx.cxsmiles(loaded[0]) == written)
print("Fe free-stereo isomer count:", len(rx.metal(written, center="Fe", stereo="free")))
print(written)
ens = rx.embed(loaded[0], n=1, seed=7)
for report in ens.check().values():
    print(report.summary())
xyzrender.render(xyzrender.load(ens.mol, rebuild=True), no_hy=True, stereo=True)

## Screen the connected chelate, not only its donor separation

The opposed pyridyl arrangements below conflict with the shared ligand/coordination distance model. The screened set keeps three arrangements; `screen=False` retains all five for explicit investigation. This screen does not prove chemical impossibility.

In [ ]:
chelate_smiles = "[Cl-]->[Ni+2]12(<-[Cl-])(<-[n]3ccccc3CCc3cccc[n]->13)<-[n]3ccccc3CCc3cccc[n]->23"
chelate_isomers = rx.metal(chelate_smiles, "OCT")
print(f"screened: {len(chelate_isomers)} arrangements | unscreened: {len(rx.metal(chelate_smiles, 'OCT', screen=False))}")
chelate_isomers.summary()
for candidate in chelate_isomers:
    ensemble = rx.embed(candidate, n=1, seed=42)
    print(rx.geom_check.check(ensemble.mol).summary())
    print("relaxed:", not ensemble.unrelaxed, "| CX match:", rx.cxsmiles(ensemble.mol) == rx.cxsmiles(candidate))

## Vacant sites: a coordination pocket

Name a geometry with more vertices than donors and the empty vertex is left as a pocket (`·` in the arrangement) for a substrate to bind later. A 3-donor Pd in a square-planar field leaves one site open.

In [ ]:
pocket = rx.metal("CC#[N]->[Pd+2](<-[Cl-])<-[Cl-]", "square_planar")  # N + 2 Cl = 3 donors, 4 vertices
pocket.summary()  # '·' marks the vacant coordination site
iso = pocket.select(index=0)  # the cis pocket (both Cl cis; the '·' vacancy sits opposite the lone Cl)
ens = rx.embed(iso).mc(preset="ensemble").minimize().prune()
for report in ens.check().values():
    print(report.summary())
xyzrender.render(ens.mol, no_hy=True, idx="s")  # the '·' vacancy sits opposite the lone Cl

### Native RDKit embedding controls

KDG with native all-in-one refinement (no experimental torsions) is the default for every molecule.
`rx.EmbedParams(native=...)` takes any RDKit `EmbedParameters` for the seed model; rxembed adds the metal
bounds and restrained UFF. Use `srETKDGv3()` for ordinary and small-ring torsions. Retain the native object
for retries; do not share it across concurrent runs.

In [ ]:
from rdkit.Chem import rdDistGeom

p = rdDistGeom.KDG()
p.useLegacyImplementation = False
native_iso = rx.metal("N->[Pt+2](<-[Cl-])(<-[Cl-])<-N", "SPL")[0]
native_ensemble = rx.embed(native_iso, n=1, params=rx.EmbedParams(seed=42, threads=1, native=p))
print(rx.geom_check.check(native_ensemble.mol, donors=native_iso.donors).summary())
print("CX match:", rx.cxsmiles(native_ensemble.mol) == rx.cxsmiles(native_iso))

### Unmodified metal graph: native DG and UFF

Use KDG + AIO directly on the real-metal graph, without edited bounds. From identical seeds, compare bond
stretching alone, native UFF with vdW off/on, then added distance or angle restraints.

`distance_k` (kcal/mol/A^2) adds soft +/-0.05 A windows around seed M-L lengths; `angle_k` (kcal/mol/deg^2)
biases the requested polyhedron, not seed angles. Neither scales native UFF, and the SP tag guides KDG but
is not a UFF restraint.

Native vdW pair parameters are `(minimum_distance, well_depth)`, not one atomic radius; switching vdW off
removes all such pairs, not just metal ones.

In [ ]:
from itertools import combinations
from math import isclose
from rdkit import Chem
from rdkit.Chem import rdForceFieldHelpers as uff
from rdkit.Chem.rdMolTransforms import GetAngleDeg, GetBondLength
from rxembed.metal_polyhedron import POLYHEDRA

native = Chem.AddHs(Chem.MolFromSmiles("CCN->[Pt@SP1+2](<-NCC)(<-[Cl-])<-[Cl-]"))
requested = next(iso for iso in rx.metal(native, "SPL") if iso.label == "cis")
requested_cx = rx.cxsmiles(requested)
p = rdDistGeom.KDG()
p.useLegacyImplementation = False
p.randomSeed = 42
p.numThreads = 1
p.embedFragmentsSeparately = False
embed_status = rdDistGeom.EmbedMolecule(native, p)
print("native KDG embed status:", embed_status, "(>=0 succeeded)")
print(rx.geom_check.check(native).summary())
print("native KDG matches the requested cis arrangement:", rx.cxsmiles(native) == requested_cx)
print("UFF types the complete graph:", uff.UFFHasAllMoleculeParams(native))
bond_params = {}
for bond in native.GetBonds():
    pair = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    bond_params[pair] = uff.GetUFFBondStretchParams(native, *pair)
missing = [pair for pair, params in bond_params.items() if params is None]
print(f"bonds without native UFF parameters: {len(missing)} of {len(bond_params)}")
ml_bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx())
            for b in native.GetBonds() if b.GetBondType() == Chem.BondType.DATIVE]
lengths = {(i, j): GetBondLength(native.GetConformer(), i, j) for i, j in ml_bonds}
angle_targets = {(requested.vertices[i], requested.metal, requested.vertices[j]): angle
                 for i, j, angle in POLYHEDRA[requested.geometry].resolved_angles}
print("Native M-L bond parameters:", {pair: bond_params[pair] for pair in ml_bonds})
paths = Chem.GetDistanceMatrix(native)
vdw_pairs = [(requested.metal, atom.GetIdx()) for atom in native.GetAtoms()
             if atom.GetAtomicNum() > 1 and paths[requested.metal, atom.GetIdx()] >= 3]
print(f"metal vdW pairs exercised: {len(vdw_pairs)}")
print("Native metal vdW (minimum distance, depth):",
      {pair: uff.GetUFFVdWParams(native, *pair) for pair in vdw_pairs})
print("donor-metal-donor | KDG angle | native UFF (coefficient, angle)")
for (i, metal), (j, other) in combinations(ml_bonds, 2):
    if metal == other:
        print((i, metal, j), round(GetAngleDeg(native.GetConformer(), i, metal, j), 1),
              uff.GetUFFAngleBendParams(native, i, metal, j))
print("model | distance k | angle k | status | max M-L change (A) | geometry OK | CX match")
for model, distance_k, angle_k in (("bonds", 0, 0), ("UFF-no-vdW", 0, 0),
                                  ("UFF", 0, 0), ("UFF", 100, 0), ("UFF", 10000, 0),
                                  ("UFF", 0, 0.01), ("UFF", 0, 0.1)):
    trial = Chem.Mol(native)
    if model == "bonds":
        ff = uff.CreateEmptyForceFieldForMol(trial)
        for (i, j), (k, r0) in bond_params.items():
            ff.UFFAddDistanceConstraint(i, j, False, r0, r0, k)
    else:
        ff = uff.UFFGetMoleculeForceField(trial, vdwThresh=0.0 if model == "UFF-no-vdW" else 10.0,
                                         ignoreInterfragInteractions=False)
    if distance_k:
        for (i, j), length in lengths.items():
            ff.UFFAddDistanceConstraint(i, j, False, length - 0.05, length + 0.05, distance_k)
    if angle_k:
        for (i, m, j), angle in angle_targets.items():
            ff.UFFAddAngleConstraint(i, m, j, False, angle, angle, angle_k)
    ff.Initialize()
    if model == "bonds":
        expected_energy = sum(k * (GetBondLength(trial.GetConformer(), *pair) - r0)**2 / 2
                              for pair, (k, r0) in bond_params.items())
        print(f"  bond-only energy check: formula {expected_energy:.6f} vs FF {ff.CalcEnergy():.6f} "
              f"(match: {isclose(ff.CalcEnergy(), expected_energy, rel_tol=1e-10, abs_tol=1e-10)})")
    status = ff.Minimize(maxIts=2000)
    drift = max(abs(GetBondLength(trial.GetConformer(), i, j) - length)
                for (i, j), length in lengths.items())
    report = rx.geom_check.check(trial)
    matches = rx.cxsmiles(trial) == requested_cx
    print(f"{model:10} | {distance_k:10g} | {angle_k:7g} | {status:6d} | {drift:18.3f} | {report.ok()!s:11} | {matches}")
    if angle_k == 0.1:
        print(report.summary())
        print("CX match:", matches)

### Controls for our distance-matrix edits

`EmbedParams(coplanar_14=...)` switches the coplanar 1–4 projections; `metal_floor_relief` switches lowering
carbon-surrogate exclusion floors to metal-aware distances. `donor_orientation` and `conjugation` switch the
optional ligand cleanup terms. All default to `True` and persist through replacement searches; a native
`EmbedParams.native` object stays independent.

In [ ]:
import numpy as np
from itertools import product
from rxembed import bounds

matrix_iso = next(iso for iso in rx.metal(
    "[Pt+2](<-[Cl-])(<-[Cl-])(<-n1ccccc1)<-n1ccccc1", "SPL") if iso.label == "cis")
cons = matrix_iso.cons
baseline_matrix = bounds._write(matrix_iso.mol, cons).bm
for label, reduced in (("coplanar 1-4", cons.copy(coplanar=[])),
                       ("metal floor relief", cons.copy(dg_floors={}))):
    changed = np.count_nonzero(bounds._write(matrix_iso.mol, reduced).bm != baseline_matrix)
    print(label, "changed entries before final smoothing:", changed)  # zero means no direct edit here

print("coplanar 1-4 | metal floor relief | geometry OK | CX match")
for coplanar_14, metal_floor_relief in product((True, False), repeat=2):
    try:
        params = rx.EmbedParams(seed=42, threads=1, coplanar_14=coplanar_14, metal_floor_relief=metal_floor_relief)
        ens = rx.embed(matrix_iso, params=params, n=1)
        report = rx.geom_check.check(ens.mol, donors=matrix_iso.donors)
        print(report.summary())
        matches = rx.cxsmiles(ens.mol) == rx.cxsmiles(matrix_iso)
        print(coplanar_14, metal_floor_relief, report.ok(), matches)
    except (ValueError, RuntimeError, TimeoutError, AssertionError) as error:
        print(coplanar_14, metal_floor_relief, type(error).__name__, str(error))

print("donor orientation | conjugation | geometry OK | CX match")
for donor_orientation, conjugation in product((True, False), repeat=2):
    try:
        params = rx.EmbedParams(seed=42, threads=1, donor_orientation=donor_orientation, conjugation=conjugation)
        ens = rx.embed(matrix_iso, params=params, n=1)
        report = rx.geom_check.check(ens.mol, donors=matrix_iso.donors)
        matches = rx.cxsmiles(ens.mol) == rx.cxsmiles(matrix_iso)
        print(donor_orientation, conjugation, report.ok(), matches)
    except (ValueError, RuntimeError, TimeoutError, AssertionError) as error:
        print(donor_orientation, conjugation, type(error).__name__, str(error))